In [1]:
!pip install roboflow ultralytics opencv-python matplotlib

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from roboflow import Roboflow
from ultralytics import YOLO
import os
import cv2
import matplotlib.pyplot as plt

In [3]:

rf = Roboflow(api_key="HJuP30s5hJePCsablmxW")
project = rf.workspace("maryygraceden-q9bha").project("waste-detection-yolo-vykag")
dataset = project.version(1).download("yolov8")


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Waste-Detection-YOLO-1 in yolov8:: 100%|██████████| 966/966 [00:02<00:00, 470.01it/s]


In [4]:
data_yaml = dataset.location + "/data.yaml"
print("Dataset YAML path:", data_yaml)

Dataset YAML path: c:\Users\reshu\Desktop\ml_Task\YOLO-Waste-detection\Waste-Detection-YOLO-1/data.yaml


In [5]:
# Load YOLOv8n pretrained model (nano, fast)
model = YOLO("yolov8n.pt")


In [6]:
# Step 7: Validation folder path
val_folder = os.path.join(dataset.location, "valid/images")
img_files = [os.path.join(val_folder, f) for f in os.listdir(val_folder) if f.endswith((".jpg", ".png"))]

In [7]:

model.train(
    data=data_yaml,      # dataset config
    epochs=25,           # reduce for faster training
    imgsz=320,           # smaller image size to speed up
    batch=16,            # batch size (adjust if VRAM allows)
    device=0,            # GPU
    name="waste_yolo_fast", # output folder
    cache=True,          # cache dataset in RAM for speed
    workers=8,           # number of CPU threads
    pretrained=True,     # use pretrained weights
    optimizer="Adam",    # faster convergence
    save_period=5,       # save every 5 epochs
)


New https://pypi.org/project/ultralytics/8.3.221 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.213  Python-3.12.5 torch-2.8.0+cpu 


ValueError: Invalid CUDA 'device=0' requested. Use 'device=cpu' or pass valid CUDA device(s) if available, i.e. 'device=0' or 'device=0,1,2,3' for Multi-GPU.

torch.cuda.is_available(): False
torch.cuda.device_count(): 0
os.environ['CUDA_VISIBLE_DEVICES']: None
See https://pytorch.org/get-started/locally/ for up-to-date torch install instructions if no CUDA devices are seen by torch.


In [ ]:
# Step 8: Function to display image + left-side text
def display_image_with_text(img, detected_classes, decomposable):
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(12, 6))

    # Left: text
    plt.subplot(1, 2, 1)
    plt.axis('off')
    plt.text(0, 0.8, "Detected Categories:", fontsize=16, fontweight='bold', color='black')
    plt.text(0, 0.6, ", ".join(detected_classes), fontsize=14, color='blue')
    plt.text(0, 0.4, f"Decomposable: {'Yes' if decomposable else 'No'}",
             fontsize=14, color='green' if decomposable else 'red')

    # Right: image
    plt.subplot(1, 2, 2)
    plt.imshow(img_rgb)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
# Step 9: Run predictions and display
for idx, img_path in enumerate(img_files):
    results = model.predict(img_path, imgsz=640, conf=0.25)
    annotated_img = results[0].plot()
    detected_classes = [model.names[int(box.cls)] for box in results[0].boxes]
    decomposable = len(set(detected_classes)) > 1
    display_image_with_text(annotated_img, detected_classes, decomposable)